In [1]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
import matplotlib.pyplot as plt
import keras
from keras import layers
from keras.applications import EfficientNetB0

IMG_SIZE = 224
BATCH_SIZE = 64

In [3]:
dataset_name = "stanford_dogs"
(ds_train, ds_test), ds_info = tfds.load(
    dataset_name, split=["train", "test"], with_info=True, as_supervised=True
)
NUM_CLASSES = ds_info.features["label"].num_classes

In [4]:
size = (IMG_SIZE, IMG_SIZE)
ds_train = ds_train.map(lambda image, label: (tf.image.resize(image, size), label))
ds_test = ds_test.map(lambda image, label: (tf.image.resize(image, size), label))

In [6]:
img_augmentation_layers = [
    layers.RandomRotation(factor=0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomFlip(),
    layers.RandomContrast(factor=0.1)
]

def img_augmentation(images):
    for layer in img_augmentation_layers:
        images = layer(images)
    return images

In [7]:
# one-hot / categical encoding
def input_preprocess_train(image, label):
    image = img_augmentation(image)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def input_preprocess_test(image, label):
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

ds_train = ds_train.map(input_preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.batch(batch_size=BATCH_SIZE, drop_remainder=True)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test.map(input_preprocess_test, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(batch_size=BATCH_SIZE, drop_remainder=True)

In [8]:
import tensorflow as tf
model = tf.keras.models.load_model("./saves/stanford_dogs_effcientnet_b0.keras")

In [9]:
# 1 epoch 10분 정도
epochs = 5
hist = model.fit(ds_train, epochs=epochs, validation_data=ds_test)

Epoch 1/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 537s 3s/step - accuracy: 0.1332 - loss: 3.6278 - val_accuracy: 0.0929 - val_loss: 3.9401
Epoch 2/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 526s 3s/step - accuracy: 0.1404 - loss: 3.5480 - val_accuracy: 0.1132 - val_loss: 3.7546
Epoch 3/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 537s 3s/step - accuracy: 0.1542 - loss: 3.4642 - val_accuracy: 0.1028 - val_loss: 3.9569
Epoch 4/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 495s 3s/step - accuracy: 0.1710 - loss: 3.3738 - val_accuracy: 0.1298 - val_loss: 3.6761
Epoch 5/5
187/187 ━━━━━━━━━━━━━━━━━━━━ 496s 3s/step - accuracy: 0.1887 - loss: 3.2889 - val_accuracy: 0.1243 - val_loss: 3.8054


In [10]:
model.save("./saves/stanford_dogs_effcientnet_b0.keras")